# Tutorial: Ejercicio 3.15 en Python - construccion del barco for dummies

Audiencia:
- Personas que quieren entender un problema de precedencias y tiempo limite.
- Personas que prefieren empezar con una busqueda simple antes de ver AMPL.

Objetivos:
- Entender que decide el modelo.
- Ver como las precedencias crean el cronograma.
- Encontrar la combinacion mas barata de tareas rapidas.


## Enunciado del ejercicio

La empresa Monkey Island debe terminar la construccion de un barco en `15` dias.

Hay `9` tareas: `A` a `I`.
Cada tarea puede hacerse de dos maneras:
- normal,
- rapida.

Hacer una tarea en modo rapido reduce su duracion, pero agrega un costo.
Ademas, existen precedencias: algunas tareas solo pueden empezar cuando otras ya terminaron.

Importante:
- la capacidad es ilimitada,
- se pueden hacer varias tareas al mismo tiempo,
- siempre que se respeten las precedencias.

Pregunta:
**¿Que tareas conviene ejecutar de forma rapida para cumplir el plazo al minimo costo?**


## Mapa del notebook

1. Ver los datos.
2. Entender la decision binaria.
3. Calcular un cronograma a partir de una decision.
4. Probar todas las combinaciones posibles.
5. Leer la solucion.
6. Probar una variante.


In [7]:
from itertools import product

TAREAS = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]
DURACION_NORMAL = {"A": 3, "B": 4, "C": 2, "D": 5, "E": 6, "F": 2, "G": 4, "H": 3, "I": 5}
DURACION_RAPIDA = {"A": 1, "B": 2, "C": 0.5, "D": 2, "E": 1, "F": 1, "G": 3, "H": 2, "I": 4}
COSTO_RAPIDO = {"A": 4, "B": 1, "C": 1, "D": 1, "E": 3, "F": 7, "G": 9, "H": 5, "I": 8}
PREDECESORES = {
    "A": [],
    "B": [],
    "C": ["A"],
    "D": ["A"],
    "E": ["B", "C"],
    "F": ["E"],
    "G": ["D"],
    "H": ["F"],
    "I": ["G"],
}
FECHA_LIMITE = 15

TAREAS


['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']

## Paso 1 - Que decide el modelo

La decision aqui es mucho mas simple que en `3.9`.

Para cada tarea solo decidimos una cosa:
- `0`: hacerla en modo normal,
- `1`: hacerla en modo rapido.

Como hay `9` tareas, eso significa que existen `2^9 = 512` combinaciones posibles.
Para una version for dummies, eso es genial, porque podemos revisarlas todas.


In [8]:
tabla_tareas = []
for tarea in TAREAS:
    tabla_tareas.append(
        {
            "tarea": tarea,
            "duracion_normal": DURACION_NORMAL[tarea],
            "duracion_rapida": DURACION_RAPIDA[tarea],
            "costo_rapida": COSTO_RAPIDO[tarea],
            "predecesores": PREDECESORES[tarea] if PREDECESORES[tarea] else ["ninguno"],
        }
    )

tabla_tareas


[{'tarea': 'A',
  'duracion_normal': 3,
  'duracion_rapida': 1,
  'costo_rapida': 4,
  'predecesores': ['ninguno']},
 {'tarea': 'B',
  'duracion_normal': 4,
  'duracion_rapida': 2,
  'costo_rapida': 1,
  'predecesores': ['ninguno']},
 {'tarea': 'C',
  'duracion_normal': 2,
  'duracion_rapida': 0.5,
  'costo_rapida': 1,
  'predecesores': ['A']},
 {'tarea': 'D',
  'duracion_normal': 5,
  'duracion_rapida': 2,
  'costo_rapida': 1,
  'predecesores': ['A']},
 {'tarea': 'E',
  'duracion_normal': 6,
  'duracion_rapida': 1,
  'costo_rapida': 3,
  'predecesores': ['B', 'C']},
 {'tarea': 'F',
  'duracion_normal': 2,
  'duracion_rapida': 1,
  'costo_rapida': 7,
  'predecesores': ['E']},
 {'tarea': 'G',
  'duracion_normal': 4,
  'duracion_rapida': 3,
  'costo_rapida': 9,
  'predecesores': ['D']},
 {'tarea': 'H',
  'duracion_normal': 3,
  'duracion_rapida': 2,
  'costo_rapida': 5,
  'predecesores': ['F']},
 {'tarea': 'I',
  'duracion_normal': 5,
  'duracion_rapida': 4,
  'costo_rapida': 8,
  'prede

## Paso 2 - Como se calcula un cronograma

Una vez elegimos que tareas van rapido, el cronograma sale por precedencias:
- si una tarea no tiene predecesores, puede empezar en `0`,
- si si tiene predecesores, empieza cuando termine el ultimo de ellos.

Como la capacidad es ilimitada, no hay que pelear por recursos. Solo importa respetar el orden.


In [9]:
def calcular_cronograma(decision_rapida, fecha_limite=FECHA_LIMITE):
    duracion = {}
    inicio = {}
    fin = {}

    for tarea in TAREAS:
        duracion[tarea] = DURACION_RAPIDA[tarea] if decision_rapida[tarea] else DURACION_NORMAL[tarea]
        if not PREDECESORES[tarea]:
            inicio[tarea] = 0
        else:
            inicio[tarea] = max(fin[pre] for pre in PREDECESORES[tarea])
        fin[tarea] = inicio[tarea] + duracion[tarea]

    duracion_total = max(fin.values())
    costo_total = sum(COSTO_RAPIDO[t] * decision_rapida[t] for t in TAREAS)

    return {
        "rapida": decision_rapida,
        "duracion": duracion,
        "inicio": inicio,
        "fin": fin,
        "duracion_total": duracion_total,
        "costo_total": costo_total,
        "cumple": duracion_total <= fecha_limite,
    }


ejemplo = calcular_cronograma({t: 0 for t in TAREAS})
{
    "duracion_total_sin_acelerar": ejemplo["duracion_total"],
    "cumple_plazo": ejemplo["cumple"],
}


{'duracion_total_sin_acelerar': 17, 'cumple_plazo': False}

## Paso 3 - Buscar la mejor combinacion

Ahora hacemos una busqueda exacta.

Como solo hay `512` combinaciones, podemos probarlas todas:
- descartamos las que no cumplen el plazo,
- y entre las factibles elegimos la de menor costo.


In [10]:
ESPERADO = {
    "costo_total": 2,
    "tareas_rapidas": {"C": 1, "D": 1},
    "duracion_total": 15,
}


def resolver_barco(fecha_limite=FECHA_LIMITE):
    mejor = None
    for bits in product([0, 1], repeat=len(TAREAS)):
        decision = {tarea: bits[i] for i, tarea in enumerate(TAREAS)}
        cronograma = calcular_cronograma(decision, fecha_limite=fecha_limite)
        if not cronograma["cumple"]:
            continue
        if mejor is None or cronograma["costo_total"] < mejor["costo_total"]:
            mejor = cronograma
    return mejor


resultado_base = resolver_barco()
assert resultado_base["costo_total"] == ESPERADO["costo_total"]
assert {t: v for t, v in resultado_base["rapida"].items() if v} == ESPERADO["tareas_rapidas"]
assert resultado_base["duracion_total"] == ESPERADO["duracion_total"]

{
    "costo_total": resultado_base["costo_total"],
    "tareas_rapidas": {t: v for t, v in resultado_base["rapida"].items() if v},
    "duracion_total": resultado_base["duracion_total"],
}


{'costo_total': 2, 'tareas_rapidas': {'C': 1, 'D': 1}, 'duracion_total': 15}

## Paso 4 - Leer la solucion

La respuesta optima es muy bonita:
- solo conviene acelerar `C` y `D`,
- el costo total extra es `2`,
- y con eso el proyecto justo termina en `15` dias.

Eso significa que no hace falta acelerar todo. Solo hay que intervenir las tareas correctas.


In [11]:
cronograma_base = []
for tarea in TAREAS:
    cronograma_base.append(
        {
            "tarea": tarea,
            "rapida": resultado_base["rapida"][tarea],
            "inicio": resultado_base["inicio"][tarea],
            "fin": resultado_base["fin"][tarea],
        }
    )

cronograma_base


[{'tarea': 'A', 'rapida': 0, 'inicio': 0, 'fin': 3},
 {'tarea': 'B', 'rapida': 0, 'inicio': 0, 'fin': 4},
 {'tarea': 'C', 'rapida': 1, 'inicio': 3, 'fin': 3.5},
 {'tarea': 'D', 'rapida': 1, 'inicio': 3, 'fin': 5},
 {'tarea': 'E', 'rapida': 0, 'inicio': 4, 'fin': 10},
 {'tarea': 'F', 'rapida': 0, 'inicio': 10, 'fin': 12},
 {'tarea': 'G', 'rapida': 0, 'inicio': 5, 'fin': 9},
 {'tarea': 'H', 'rapida': 0, 'inicio': 12, 'fin': 15},
 {'tarea': 'I', 'rapida': 0, 'inicio': 9, 'fin': 14}]

## Ejercicio guiado

Baja el plazo de `15` a `14` dias y mira que pasa.

Pregunta previa:
**si el plazo se aprieta un poco mas, alcanza con acelerar las mismas tareas o hay que cambiar la combinacion?**


In [12]:
resultado_due_14 = resolver_barco(fecha_limite=14)

{
    "costo_con_due_15": resultado_base["costo_total"],
    "tareas_rapidas_con_due_15": {t: v for t, v in resultado_base["rapida"].items() if v},
    "costo_con_due_14": resultado_due_14["costo_total"],
    "tareas_rapidas_con_due_14": {t: v for t, v in resultado_due_14["rapida"].items() if v},
}


{'costo_con_due_15': 2,
 'tareas_rapidas_con_due_15': {'C': 1, 'D': 1},
 'costo_con_due_14': 4,
 'tareas_rapidas_con_due_14': {'D': 1, 'E': 1}}

## Comparacion con el libro

El libro reporta que la solucion optima acelera solo las tareas `C` y `D`, con costo total `2`, cumpliendo el plazo de `15` dias.

La comparacion importante es esa decision y ese costo. Los tiempos de inicio exactos pueden escribirse de varias maneras equivalentes si se mantiene la factibilidad y el mismo costo minimo.


In [13]:
comparacion_libro = {
    "costo_coincide": resultado_base["costo_total"] == ESPERADO["costo_total"],
    "tareas_rapidas_coinciden": {t: v for t, v in resultado_base["rapida"].items() if v} == ESPERADO["tareas_rapidas"],
    "duracion_total_coincide": resultado_base["duracion_total"] == ESPERADO["duracion_total"],
    "conclusion": "La respuesta coincide con la del libro en costo y tareas aceleradas.",
}

comparacion_libro


{'costo_coincide': True,
 'tareas_rapidas_coinciden': True,
 'duracion_total_coincide': True,
 'conclusion': 'La respuesta coincide con la del libro en costo y tareas aceleradas.'}